# 📝 Edu Question Generator — Google Colab

**الخطوات قبل التشغيل:**
1. احصلي على مفتاح OpenAI من: https://platform.openai.com/api-keys
2. شغّلي الخلايا بالترتيب (Runtime → Run all)
3. أدخلي المفتاح في الخلية الثانية
4. ارفعي ملف PDF أو Word أو TXT
5. حمّلي Excel من آخر خلية

**المسار:** استخراج النص → تقسيم → GPT → أسئلة → Excel

In [ ]:
!pip install -q openai pdfplumber python-docx langdetect pandas openpyxl

In [ ]:
import json
import math
import os
import re
from io import BytesIO

import pandas as pd
import pdfplumber
import docx
from google.colab import files
from langdetect import detect
from openai import OpenAI
from getpass import getpass

# ── الإعدادات ──
OPENAI_API_KEY = getpass("🔑 أدخلي OpenAI API Key: ")
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1"          # gpt-4.1 | gpt-4o | o3
DIFFICULTY = "Hard"        # Easy | Medium | Hard
NUM_QUESTIONS = 10
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
MAX_SEGMENTS = 12
QUESTION_TYPES = ["mcq", "tf", "short"]  # mcq | tf | short

print(f"✅ جاهز — النموذج: {MODEL} | الصعوبة: {DIFFICULTY}")

In [ ]:
def load_text(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        with pdfplumber.open(path) as pdf:
            return "\n".join(p.extract_text() or "" for p in pdf.pages)
    if ext in {".docx", ".doc"}:
        doc = docx.Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()


def chunk_text(text: str, size=800, overlap=150):
    chunks, step = [], max(size - overlap, 1)
    for i in range(0, len(text), step):
        piece = text[i:i + size].strip()
        if piece:
            chunks.append(piece)
    return chunks


def group_chunks(chunks, max_groups=12):
    if len(chunks) <= max_groups:
        return chunks
    group_size = math.ceil(len(chunks) / max_groups)
    return ["\n\n".join(chunks[i:i + group_size]) for i in range(0, len(chunks), group_size)]


def detect_lang(text: str) -> str:
    try:
        return "ar" if detect(text) == "ar" else "en"
    except Exception:
        return "en"


def build_prompt(context, lang, difficulty, types, num_questions):
    dl_extra = ""
    if difficulty == "Hard":
        dl_extra = (
            "\n- Include calculation-based questions when content supports it (backprop, matrix dims, gradients, CNN)."
            if lang == "en" else
            "\n- أدرج أسئلة حسابية عندما يناسب المحتوى (Backprop، أبعاد المصفوفات، CNN)."
        )
    if lang == "ar":
        return f"""أنت خبير تعليمي. أنشئ {num_questions} سؤالاً من المحتوى.
الصعوبة: {difficulty} | الأنواع: {', '.join(types)}{dl_extra}
استخدم Bloom's Taxonomy. أعد JSON فقط.
المحتوى:\n{context}
تنسيق: {{"mcq":[{{"q":"","options":["","","",""],"answer":"","difficulty":"{difficulty}"}}], "tf":[{{"q":"","answer":true,"difficulty":"{difficulty}"}}], "short":[{{"q":"","answer":"","difficulty":"{difficulty}"}}]}}
أدرج فقط: {', '.join(types)}"""
    return f"""Generate {num_questions} academic questions from the content.
Difficulty: {difficulty} | Types: {', '.join(types)}{dl_extra}
Use Bloom's Taxonomy. Return JSON only.
Content:\n{context}
Format: {{"mcq":[{{"q":"","options":["","","",""],"answer":"","difficulty":"{difficulty}"}}], "tf":[{{"q":"","answer":true,"difficulty":"{difficulty}"}}], "short":[{{"q":"","answer":"","difficulty":"{difficulty}"}}]}}
Include only: {', '.join(types)}"""


def safe_json(raw: str) -> dict:
    text = raw.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def call_gpt(context, lang, difficulty, types, num_questions, model):
    prompt = build_prompt(context, lang, difficulty, types, num_questions)
    system = "Return valid JSON only." if lang == "en" else "أعد JSON صالحاً فقط."
    resp = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=0.4,
    )
    return safe_json(resp.choices[0].message.content or "{}")


def to_dataframe(payload, default_difficulty="Medium"):
    rows, n = [], 1
    for item in payload.get("mcq", []):
        opts = item.get("options", [])
        rows.append({"#": n, "Type": "MCQ", "Question": item.get("q", ""),
            "Option A": opts[0] if len(opts)>0 else "", "Option B": opts[1] if len(opts)>1 else "",
            "Option C": opts[2] if len(opts)>2 else "", "Option D": opts[3] if len(opts)>3 else "",
            "Answer": item.get("answer", ""), "Difficulty": item.get("difficulty", default_difficulty)})
        n += 1
    for item in payload.get("tf", []):
        ans = item.get("answer"); ans = "True" if ans is True else "False" if ans is False else str(ans)
        rows.append({"#": n, "Type": "True/False", "Question": item.get("q", ""),
            "Option A": "True", "Option B": "False", "Option C": "", "Option D": "",
            "Answer": ans, "Difficulty": item.get("difficulty", default_difficulty)})
        n += 1
    for item in payload.get("short", []):
        rows.append({"#": n, "Type": "Short Answer", "Question": item.get("q", ""),
            "Option A": "", "Option B": "", "Option C": "", "Option D": "",
            "Answer": item.get("answer", ""), "Difficulty": item.get("difficulty", default_difficulty)})
        n += 1
    cols = ["#", "Type", "Question", "Option A", "Option B", "Option C", "Option D", "Answer", "Difficulty"]
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)

print("✅ الدوال جاهزة")

In [ ]:
print("📂 ارفعي ملف PDF أو DOCX أو TXT:")
uploaded = files.upload()
filename = next(iter(uploaded))
with open(filename, "wb") as f:
    f.write(uploaded[filename])

text = load_text(filename).strip()
lang = detect_lang(text)
chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
segments = group_chunks(chunks, MAX_SEGMENTS)

print(f"📄 الملف: {filename}")
print(f"🌐 اللغة: {lang} | 🔢 عدد الأحرف: {len(text):,} | 📦 المقاطع: {len(segments)}")

In [ ]:
merged = {"mcq": [], "tf": [], "short": []}

if len(segments) == 1:
    payload = call_gpt(segments[0], lang, DIFFICULTY, QUESTION_TYPES, NUM_QUESTIONS, MODEL)
    merged = payload
else:
    base = max(1, NUM_QUESTIONS // len(segments))
    remainder = max(0, NUM_QUESTIONS - base * len(segments))
    for i, seg in enumerate(segments):
        count = base + (1 if i < remainder else 0)
        print(f"⏳ مقطع {i+1}/{len(segments)} — {count} سؤال...")
        p = call_gpt(seg, lang, DIFFICULTY, QUESTION_TYPES, count, MODEL)
        for k in merged:
            merged[k].extend(p.get(k, []))

df = to_dataframe(merged, DIFFICULTY)
print(f"\n✅ تم توليد {len(df)} سؤال")
df

In [ ]:
out_name = os.path.splitext(filename)[0] + "_questions.xlsx"
df.to_excel(out_name, index=False, engine="openpyxl")
files.download(out_name)
print(f"📥 تم تحميل: {out_name}")